# 7.5 Compute Resources

FLOP estimator, GPU memory calculator, Chinchilla scaling law, and training time projections.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def count_params(d, h, L, V=50000, ffn=4): return V*d + L*(4*d*d + 2*ffn*d*d)
def mem_gb(n, d, bs=1, seq=2048, dt=2): return (n*dt+n*dt+n*8+bs*seq*d*dt*12)/1e9
def train_hrs(N, mfu=0.4, D_mult=20): return 6*N*D_mult*N/(312e12*mfu*3600)

configs = [
    {'name':'125M', 'd':768, 'h':12, 'L':12},
    {'name':'350M', 'd':1024,'h':16, 'L':24},
    {'name':'1.3B', 'd':2048,'h':16, 'L':24},
    {'name':'6.7B', 'd':4096,'h':32, 'L':32},
]
print(f'{"Model":8s} {"Params(M)":12s} {"Mem(GB)":10s} {"Hours":8s}')
for c in configs:
    N=count_params(c['d'],c['h'],c['L'])
    print(f"{c['name']:8s} {N/1e6:12.1f} {mem_gb(N,c['d']):10.2f} {train_hrs(N):8.1f}")

In [ ]:
ns = np.logspace(7,11,80)
mfu_vals = np.linspace(0.1,0.9,50)
hrs_mfu = [train_hrs(1.3e9, mfu=m) for m in mfu_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(ns/1e9, 20*ns/1e9, color='mediumseagreen', lw=2)
axes[0].set(xlabel='Model (B params)', ylabel='Optimal Tokens (B)', title='Chinchilla Scaling')
axes[0].grid(True, alpha=0.3)

axes[1].plot(mfu_vals, hrs_mfu, color='steelblue', lw=2)
axes[1].axvline(0.4, color='red', linestyle='--', label='MFU=0.4')
axes[1].set(xlabel='MFU', ylabel='Training Hours', title='1.3B: Training Time vs MFU')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/compute_resources.png')
print('Saved compute_resources.png')